# 🏥 Hospital Readmission Prediction
## Predicting 30-Day Readmission in Diabetic Patients
**Dataset:** UCI Diabetes 130-US Hospitals | 98,000+ patient records  
**Goal:** Identify patients at risk of readmission within 30 days using unsupervised and supervised machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly as plt

# Loading data
df = pd.read_csv('archive/diabetic_data.csv')

print(df.shape)
print(df.head())
print(df.dtypes)

(101766, 50)
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)      ?   
1        149190     55629189        Caucasian  Female  [10-20)      ?   
2         64410     86047875  AfricanAmerican  Female  [20-30)      ?   
3        500364     82442376        Caucasian    Male  [30-40)      ?   
4         16680     42519267        Caucasian    Male  [40-50)      ?   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   
3                  1                         1                    7   
4                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburide-metformin  \
0                 1  ...          No      No        

## 1. Data Loading & Cleaning
The dataset contains 50 features including demographics, diagnoses, medications, and hospital visit history. Missing values were encoded as `?` and required replacement. Columns with >40% missing data were dropped.

In [4]:
# Looking into data
# replace ? vals with Nan so pyhton can recognize
df.replace('?', np.nan, inplace=True)


# How many missing values per column?
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

# What does target column look like?
print("\nReadmitted value counts:")
print(df['readmitted'].value_counts())

weight               98569
max_glu_serum        96420
A1Cresult            84748
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64

Readmitted value counts:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


In [5]:
# Drop columns with too many missing values
cols_to_drop = ['weight', 'max_glu_serum', 'A1Cresult', 
                'medical_specialty', 'payer_code']
df.drop(columns=cols_to_drop, inplace=True)


# Drop remaining rows with missing values
df.dropna(inplace=True)

# Convert target to binary: <30 days = 1, everything else = 0
df['readmitted'] = (df['readmitted'] == '<30').astype(int)

print(df.shape)
print(df['readmitted'].value_counts())

(98053, 45)
readmitted
0    86987
1    11066
Name: count, dtype: int64


In [6]:
# Convert age brackets to numeric midpoints/means
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df['age'] = df['age'].map(age_map)

# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove target from these lists
numeric_cols.remove('readmitted')

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print(categorical_cols)

Numeric columns: 14
Categorical columns: 30
['race', 'gender', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


/var/folders/rm/t5gjw3s17t9cldkd_4nn32jc0000gp/T/ipykernel_2127/3076782994.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


In [7]:
# Fix the pandas warning going forward
categorical_cols = df.select_dtypes(include=['str', 'object']).columns.tolist()

# Simplify diagnosis codes into broad categories
def map_diag(code):
    try:
        code = str(code)
        c = float(code)
        if 390 <= c <= 459 or c == 785: return 'Circulatory'
        if 460 <= c <= 519 or c == 786: return 'Respiratory'
        if 520 <= c <= 579 or c == 787: return 'Digestive'
        if code.startswith('250'): return 'Diabetes'
        if 800 <= c <= 999: return 'Injury'
        if 710 <= c <= 739: return 'Musculoskeletal'
        if 580 <= c <= 629 or c == 788: return 'Genitourinary'
        if 140 <= c <= 239: return 'Neoplasms'
        return 'Other'
    except:
        return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(map_diag)

# One-hot encode all categorical columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(df_encoded.shape)

(98053, 94)


In [8]:
from sklearn.preprocessing import StandardScaler

# Separate features from target
X = df_encoded.drop(columns=['readmitted', 'encounter_id', 'patient_nbr'])
y = df_encoded['readmitted']

# Scale features (important for K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (98053, 91)
Target shape: (98053,)


## 2. Patient Segmentation (K-Means Clustering)
Before building a classifier, we explore whether natural patient groups exist in the data. K-Means clustering groups similar patients together without using the readmission label.

In [22]:
from sklearn.cluster import KMeans

# Try K from 2 to 10 and measure inertia
inertia = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

# Plot the elbow
import plotly.express as px

fig = px.line(
    x=list(k_range), 
    y=inertia,
    markers=True,
    title='Elbow Method - Finding Optimal K',
    labels={'x': 'Number of Clusters (K)', 'y': 'Inertia'}
)
fig.show()

In [10]:
# Fit final K-Means with K=4
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
df_encoded['cluster'] = km_final.fit_predict(X_scaled)

# See how many patients in each cluster
print(df_encoded['cluster'].value_counts().sort_index())

# See readmission rate per cluster - this is the key insight
cluster_analysis = df_encoded.groupby('cluster')['readmitted'].agg(['mean', 'count'])
cluster_analysis.columns = ['readmission_rate', 'patient_count']
cluster_analysis['readmission_rate'] = (cluster_analysis['readmission_rate'] * 100).round(2)
print("\nReadmission rate per cluster:")
print(cluster_analysis)

cluster
0    11201
1     6118
2    48243
3    32491
Name: count, dtype: int64

Readmission rate per cluster:
         readmission_rate  patient_count
cluster                                 
0                   11.43          11201
1                   10.51           6118
2                   12.34          48243
3                    9.82          32491


In [11]:
from sklearn.decomposition import PCA

# Reduce to 2D for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

import plotly.express as px

fig = px.scatter(
    x=X_pca[:, 0], 
    y=X_pca[:, 1],
    color=df_encoded['cluster'].astype(str),
    title='Patient Clusters (PCA 2D View)',
    labels={'x': 'PCA Component 1', 'y': 'PCA Component 2', 'color': 'Cluster'},
    opacity=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

print(f"Variance explained by 2 components: {pca.explained_variance_ratio_.sum()*100:.1f}%")

Variance explained by 2 components: 6.3%


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from imblearn.over_sampling import SMOTE

# Add cluster as a feature
X = df_encoded.drop(columns=['readmitted', 'encounter_id', 'patient_nbr'])
y = df_encoded['readmitted']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_sm.value_counts().to_dict())

Before SMOTE: {0: 69589, 1: 8853}
After SMOTE: {0: 69589, 1: 69589}


## 3. Readmission Prediction (Random Forest Classifier)
Using patient features plus cluster labels, we train a binary classifier to predict 30-day readmission. Class imbalance (11% positive rate) is addressed using balanced class weights and a tuned decision threshold.

In [14]:
from sklearn.ensemble import RandomForestClassifier

# Baseline: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sm, y_train_sm)
lr_preds = lr.predict(X_test)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])

print("=== Logistic Regression ===")
print(classification_report(y_test, lr_preds))
print(f"AUC-ROC: {lr_auc:.4f}")

# Main model: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
rf_preds = rf.predict(X_test)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

print("\n=== Random Forest ===")
print(classification_report(y_test, rf_preds))
print(f"AUC-ROC: {rf_auc:.4f}")

/Users/samanvitaanand/Library/CloudStorage/OneDrive-CafeStreamLLC/Code/ML/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.89      0.92      0.91     17398
           1       0.16      0.12      0.14      2213

    accuracy                           0.83     19611
   macro avg       0.53      0.52      0.52     19611
weighted avg       0.81      0.83      0.82     19611

AUC-ROC: 0.5560

=== Random Forest ===
              precision    recall  f1-score   support

           0       0.89      0.97      0.93     17398
           1       0.17      0.04      0.07      2213

    accuracy                           0.87     19611
   macro avg       0.53      0.51      0.50     19611
weighted avg       0.81      0.87      0.83     19611

AUC-ROC: 0.5822


In [15]:
# Random Forest with class weight balanced
rf2 = RandomForestClassifier(
    n_estimators=100, 
    random_state=42, 
    n_jobs=-1,
    class_weight='balanced'
)
rf2.fit(X_train, y_train)  # use original imbalanced training data this time
rf2_probs = rf2.predict_proba(X_test)[:, 1]
rf2_auc = roc_auc_score(y_test, rf2_probs)

# Lower threshold to 0.3
rf2_preds = (rf2_probs >= 0.3).astype(int)

print("=== Random Forest (balanced + threshold 0.3) ===")
print(classification_report(y_test, rf2_preds))
print(f"AUC-ROC: {rf2_auc:.4f}")

=== Random Forest (balanced + threshold 0.3) ===
              precision    recall  f1-score   support

           0       0.91      0.81      0.85     17398
           1       0.19      0.37      0.25      2213

    accuracy                           0.76     19611
   macro avg       0.55      0.59      0.55     19611
weighted avg       0.83      0.76      0.79     19611

AUC-ROC: 0.6384


In [16]:
import pandas as pd

# Get feature importances
feature_names = X.columns
importances = rf2.feature_importances_

feat_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False).head(15)

fig = px.bar(
    feat_df,
    x='importance',
    y='feature',
    orientation='h',
    title='Top 15 Features Predicting 30-Day Readmission',
    labels={'importance': 'Feature Importance', 'feature': ''},
    color='importance',
    color_continuous_scale='Teal'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [19]:
from sklearn.metrics import roc_curve

# Get ROC curve data
fpr, tpr, _ = roc_curve(y_test, rf2_probs)

import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', 
                          name=f'Random Forest (AUC = {rf2_auc:.4f})',
                          line=dict(color='teal', width=2)))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                          name='Random (AUC = 0.50)',
                          line=dict(color='gray', dash='dash')))
fig.update_layout(
    title='ROC Curve — 30-Day Readmission Prediction',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=650, height=500
)
fig.show()


## 4. Key Findings
- Top predictors of readmission: number of lab procedures, medications, time in hospital, age, and prior inpatient visits
- A naive model caught only 4% of readmissions. With threshold tuning, recall improved to 37%
- Clinically, this model helps flag high-risk patients before discharge for targeted follow-up